# EXP-02 · Databricks + Apache Spark — carga y consulta a 582, 5 000 y 10 000 filas
**Proyecto:** Sistema automatizado para el monitoreo de empleados (BASAV · Grupo 2)

Este notebook sigue la guía de pasos del encargado (pasos 4 a 19). Cada celda que lleva **CAPTURA D-xx** imprime al inicio
el código de captura, la fecha y hora (Lima) y el responsable, para que queden visibles en la imagen. El código **E-xx** lo asigna el encargado.

| Paso guía | Qué hace | Captura | Punto del informe |
|---|---|---|---|
| 4–6 | Esquema, volumen y directorio | D-01 | 5.3 paso 1 |
| 7 | Subir los 6 CSV (por la interfaz) y listarlos | D-02 | 5.3 paso 2 |
| 8 | `%fs head` de cada archivo | D-03 | 5.3 paso 2 |
| 9 | DataFrames `dfAsistencias` y `dfTareas` | D-04 | 5.3 paso 2 |
| 10 | Visualización con `display()` | D-05 | 3.5.5 y 6.4 |
| 11 | Tablas `tb_asistencias` y `tb_tareas` | D-06 | 5.3 paso 2 |
| 12 | COUNT + GROUP BY (+ KPI de la línea base) | D-07 | 5.3 paso 4 y 6.5 |
| 13 | Pasos 7–12 con 5 000 y 10 000 | D-08… | 5.3 pasos 3 y 4 |
| 14 | Edición, compute, runtime y Spark | D-09 | 4.4 y 3.5.4 |
| 15–17 | Mediciones (1 calentamiento + 5), CSV de tiempos, resumen y registro RUN | captura del CSV | 4.3, 4.6, 6.2, 6.3 |
| 18 | Gráfico tiempo vs. filas | — | 3.5.5 y 6.4 |
| 19 | Descripción de lo observado | — | 6.5 |

**Antes de empezar:** ejecuta `generar_sinteticos.py` en tu PC para obtener los 6 CSV y el `manifiesto_datos.csv`.

## 0. Configuración (completar `RESPONSABLE` y revisar el catálogo)

In [ ]:
import time, os, platform
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

RESPONSABLE = "Luis Obregón Hidalgo"          # aparece en cada captura
EDICION     = "Free Edition"             # confirmar en la cuenta (paso 14)
CATALOGO    = "workspace"                # en Free Edition el catálogo por defecto es 'workspace'
ESQUEMA     = "bd_rendimiento"
VOLUMEN     = "vol_rendimiento"
DIRECTORIO  = "datos_base"
REPETICIONES  = 5                        # repeticiones que se anotan
CALENTAMIENTO = 1                        # ejecuciones previas que NO se anotan
RUN_INICIAL   = 3                        # RUN-01 y RUN-02 ya están en el informe (EXP-01)

RUTA_VOL   = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"
RUTA_DATOS = f"{RUTA_VOL}/{DIRECTORIO}"
RUTA_RES   = f"{RUTA_VOL}/resultados"
TZ = ZoneInfo("America/Lima")
ESPERADOS = [f"{t}_{e}.csv" for e in ("base", "5000", "10000") for t in ("asistencias", "tareas")]

# Escalas: etiqueta -> sufijo de archivo y de tabla
ESCALAS = [
    {"filas": 582,   "archivo": "base",  "sufijo": ""},
    {"filas": 5000,  "archivo": "5000",  "sufijo": "_5000"},
    {"filas": 10000, "archivo": "10000", "sufijo": "_10000"},
]

try:
    NOTEBOOK = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
except Exception:
    NOTEBOOK = "EXP02_Databricks_Rendimiento"

def ahora():
    return datetime.now(TZ).strftime("%Y-%m-%d %H:%M:%S")

def marca(captura):
    print(f"[{captura}] {ahora()} (Lima) | Responsable: {RESPONSABLE} | Notebook: {NOTEBOOK}")
    print("-" * 90)

def tabla(nombre_base, sufijo):
    return f"{CATALOGO}.{ESQUEMA}.{nombre_base}{sufijo}"

print("Configuración lista:", RUTA_DATOS)

## Pasos 4, 5 y 6 · Esquema, volumen y directorio — **CAPTURA D-01** (5.3, paso 1)

In [ ]:
marca("D-01")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")                    # paso 4
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")          # paso 5
dbutils.fs.mkdirs(RUTA_DATOS)                                                     # paso 6
dbutils.fs.mkdirs(RUTA_RES)

print("Esquema :", f"{CATALOGO}.{ESQUEMA}")
print("Volumen :", f"{CATALOGO}.{ESQUEMA}.{VOLUMEN}")
print("Carpeta :", RUTA_DATOS)
display(spark.sql(f"SHOW VOLUMES IN {CATALOGO}.{ESQUEMA}"))
display(dbutils.fs.ls(RUTA_VOL))

## Paso 7 · Subir los CSV — **CAPTURA D-02** (5.3, paso 2)
**Opción A (interfaz):** Catalog → workspace → bd_rendimiento → vol_rendimiento → datos_base → *Upload to this volume*, y subir los 6 CSV
y `manifiesto_datos.csv`. **Opción B (scripts del repositorio):** la primera celda siguiente ejecuta `reconstruir_base_desde_repo.py` y
`generar_sinteticos.py` (carpeta `scripts/` junto al notebook) y escribe los CSV en el volumen. **Opción C:** copiarlos desde GitHub.
Usar solo una opción. Los CSV son deterministas (semilla 2026): el SHA-256 debe coincidir con `datos/manifiesto_datos.csv` del repositorio.
Después, la celda D-02 comprueba que estén todos y que su SHA-256 coincida con el manifiesto del generador.

In [ ]:
# Opción B: generar los CSV en Databricks con los scripts del repositorio (scripts/ junto a este notebook)
import subprocess, sys, shutil
CARPETA_NB = os.path.dirname("/Workspace" + NOTEBOOK) if not NOTEBOOK.startswith("/Workspace") else os.path.dirname(NOTEBOOK)
SCRIPTS = next((d for d in (os.path.join(CARPETA_NB, "scripts"), os.path.join(os.path.dirname(CARPETA_NB), "scripts")) if os.path.isdir(d)), "")
def correr(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if r.returncode != 0:
        raise RuntimeError("Falló " + args[0])
if not SCRIPTS:
    print("Opción B omitida: no hay carpeta scripts/ junto al notebook.")
else:
    correr(f"{SCRIPTS}/reconstruir_base_desde_repo.py", "--salida", "/tmp/export_base")
    correr(f"{SCRIPTS}/generar_sinteticos.py", "--asistencia", "/tmp/export_base/Asistencia.csv",
           "--tareas", "/tmp/export_base/Tareas.csv", "--semilla", "2026", "--salida", "/tmp/datos_generados")
    for a in ESPERADOS + ["manifiesto_datos.csv", "manifiesto_datos.json"]:
        shutil.copyfile(f"/tmp/datos_generados/{a}", f"{RUTA_DATOS}/{a}")
    print("CSV escritos en", RUTA_DATOS)

In [ ]:
# Opción C: copiar los CSV desde el repositorio del grupo al volumen (omitir si se subieron a mano)
import urllib.request
REPO = "luishidalgof15-glitch/Sistema_Remuneraciones_Reportes_Luis"   # fork con la rama de esta etapa
RAMA = "feature/databricks-exp02"
URL_DATOS = f"https://raw.githubusercontent.com/{REPO}/{RAMA}/06_Big_Data_Databricks/datos"
ya = {f.name for f in dbutils.fs.ls(RUTA_DATOS)}
for a in ESPERADOS + ["manifiesto_datos.csv"]:
    if a in ya:
        print("ya existe (no se copia):", a)
        continue
    urllib.request.urlretrieve(f"{URL_DATOS}/{a}", f"{RUTA_DATOS}/{a}")
    print("copiado desde GitHub:", a)

In [ ]:
marca("D-02")
import hashlib
archivos = {f.name: f.size for f in dbutils.fs.ls(RUTA_DATOS)}
faltan = [a for a in ESPERADOS if a not in archivos]
if faltan:
    raise FileNotFoundError(f"Faltan archivos en {RUTA_DATOS}: {faltan}")
manif = pd.read_csv(f"{RUTA_DATOS}/manifiesto_datos.csv").set_index("archivo") if "manifiesto_datos.csv" in archivos else None
filas = []
for a in ESPERADOS:
    sha = hashlib.sha256(open(f"{RUTA_DATOS}/{a}", "rb").read()).hexdigest()
    filas.append({"archivo": a, "bytes": archivos[a],
                  "sha256_coincide_manifiesto": None if manif is None else sha == manif.loc[a, "sha256"]})
display(pd.DataFrame(filas))

## Paso 8 · Contenido de cada archivo con `%fs head` — **CAPTURA D-03** (5.3, paso 2)
Si cambiaste el catálogo, el esquema o el volumen, corrige la ruta de estas dos celdas (`%fs` no acepta variables).
Si `--maxBytes=600` da error en tu workspace, bórralo y deja solo `%fs head <ruta>`.

In [ ]:
%fs head --maxBytes=600 /Volumes/workspace/bd_rendimiento/vol_rendimiento/datos_base/asistencias_base.csv

In [ ]:
%fs head --maxBytes=600 /Volumes/workspace/bd_rendimiento/vol_rendimiento/datos_base/tareas_base.csv

## Paso 9 · DataFrames de Spark `dfAsistencias` y `dfTareas` — **CAPTURA D-04** (5.3, paso 2)

In [ ]:
marca("D-04")
ESQ_ASIS = StructType([
    StructField("ID_Asistencia", IntegerType()), StructField("ID_Trabajador", IntegerType()),
    StructField("Fecha_Entrada", StringType()),  StructField("Fecha_Salida", StringType()),
    StructField("Fecha_Calculada", StringType()), StructField("Estado_Asistencia", StringType()),
])
ESQ_TAREA = StructType([
    StructField("ID_Tarea", IntegerType()), StructField("ID_Trabajador", IntegerType()),
    StructField("ID_Proyecto", IntegerType()), StructField("Fecha", StringType()),
    StructField("Fecha_Inicio", StringType()), StructField("Fecha_Entrega", StringType()),
    StructField("Estado_Tarea", StringType()), StructField("Observaciones", StringType()),
])

def leer(tipo, archivo):
    """Lee un CSV del volumen con esquema explícito y convierte las fechas a timestamp."""
    esquema = ESQ_ASIS if tipo == "asistencias" else ESQ_TAREA
    fechas = ["Fecha_Entrada", "Fecha_Salida", "Fecha_Calculada"] if tipo == "asistencias" \
             else ["Fecha", "Fecha_Inicio", "Fecha_Entrega"]
    df = (spark.read.format("csv").option("header", "true").schema(esquema)
          .load(f"{RUTA_DATOS}/{tipo}_{archivo}.csv"))
    for c in fechas:
        df = df.withColumn(c, F.expr(f"try_cast({c} AS TIMESTAMP)"))
    return df

dfAsistencias = leer("asistencias", "base")
dfTareas      = leer("tareas", "base")

dfAsistencias.printSchema()
dfTareas.printSchema()
print("Filas dfAsistencias:", dfAsistencias.count(), "| Filas dfTareas:", dfTareas.count())
display(dfAsistencias.limit(5))
display(dfTareas.limit(5))

## Paso 10 · Visualización con `display()` — **CAPTURA D-05** (3.5.5 y 6.4)
Después de ejecutar: pulsar **+ → Visualization → Bar**. X = `mes`, Y = `cantidad` (Sum), Group by = `Estado_Asistencia`.
Ponerle título al gráfico.

In [ ]:
marca("D-05")
display(
    dfAsistencias
      .withColumn("mes", F.date_format("Fecha_Entrada", "yyyy-MM"))
      .groupBy("mes", "Estado_Asistencia").agg(F.count("*").alias("cantidad"))
      .orderBy("mes", "Estado_Asistencia")
)

**Explicación de la visualización (2 o 3 líneas, escribir después de ver el gráfico):**

[Escribir aquí qué muestra el gráfico: cuántas asistencias hay por mes y qué parte son tardanzas. Usar solo lo que se ve.]

## Paso 11 · Guardar `tb_asistencias` y `tb_tareas` en el esquema — **CAPTURA D-06** (5.3, paso 2)

In [ ]:
marca("D-06")
dfAsistencias.write.mode("overwrite").saveAsTable(tabla("tb_asistencias", ""))
dfTareas.write.mode("overwrite").saveAsTable(tabla("tb_tareas", ""))
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

## Paso 12 · COUNT + GROUP BY ordenado de mayor a menor — **CAPTURA D-07** (5.3, paso 4 y 6.5)

In [ ]:
CONSULTAS = {
    "asistencias": "SELECT Estado_Asistencia, COUNT(*) AS cantidad FROM {t} "
                   "GROUP BY Estado_Asistencia ORDER BY cantidad DESC",
    "tareas":      "SELECT Estado_Tarea, COUNT(*) AS cantidad FROM {t} "
                   "GROUP BY Estado_Tarea ORDER BY cantidad DESC",
}

def consultas_y_kpi(sufijo, captura):
    marca(captura)
    ta, tt = tabla("tb_asistencias", sufijo), tabla("tb_tareas", sufijo)
    display(spark.sql(CONSULTAS["asistencias"].format(t=ta)))
    display(spark.sql(CONSULTAS["tareas"].format(t=tt)))
    # KPI con las mismas reglas del Data Mart (3.4.1): se suman numeradores y denominadores
    kpi = spark.sql(f"""
        WITH a AS (SELECT COUNT(*) AS asistencias,
                          SUM(CASE WHEN lower(Estado_Asistencia) = 'a tiempo' THEN 1 ELSE 0 END) AS puntuales
                   FROM {ta}),
             t AS (SELECT COUNT(*) AS tareas,
                          SUM(CASE WHEN lower(Estado_Tarea) = 'completada'
                                    AND (Observaciones IS NULL OR trim(Observaciones) = '') THEN 1 ELSE 0 END) AS sin_obs,
                          SUM(CASE WHEN upper(Observaciones) LIKE '%FUERA DE PLAZO%' THEN 1 ELSE 0 END) AS fuera_plazo
                   FROM {tt})
        SELECT asistencias, puntuales, tareas, sin_obs, fuera_plazo,
               ROUND(100.0 * puntuales / asistencias, 2) AS TPA,
               ROUND(100.0 * sin_obs / tareas, 2)        AS ICT,
               ROUND(100.0 * fuera_plazo / tareas, 2)    AS TDT,
               ROUND(0.30 * (100.0 * puntuales / asistencias) + 0.50 * (100.0 * sin_obs / tareas)
                     + 0.20 * (100 - 100.0 * fuera_plazo / tareas), 2) AS IEP
        FROM a CROSS JOIN t""")
    display(kpi)
    return kpi.toPandas().iloc[0].to_dict()

kpi_base = consultas_y_kpi("", "D-07")
print("Referencia BI (Power BI): TPA 92,10 % | ICT 56,36 % | TDT 3,44 % | IEP 75,12")

**Qué significa el resultado (escribir después de ejecutar):**

[Ej.: cuántas asistencias son "A tiempo" y cuántas "Tardanza"; cuántas tareas hay en cada estado; si los KPI de la línea base coinciden con Power BI.]

## Paso 13 · Repetir los pasos 7–12 con 5 000 y 10 000 filas — **CAPTURAS D-08 en adelante** (5.3, pasos 3 y 4)
Cada escala imprime su propio código de captura. Si el encargado asigna otros números, cambia la lista `CAPTURAS_ESCALA`.

In [ ]:
CAPTURAS_ESCALA = {"5000": "D-08", "10000": "D-08b"}
kpi_por_escala = {582: kpi_base}

for e in ESCALAS[1:]:
    cap = CAPTURAS_ESCALA[e["archivo"]]
    marca(f"{cap} · {e['filas']} filas · head + DataFrames + tablas")
    for tipo in ("asistencias", "tareas"):
        print(f"--- {tipo}_{e['archivo']}.csv (primeros bytes) ---")
        print(dbutils.fs.head(f"{RUTA_DATOS}/{tipo}_{e['archivo']}.csv", 400))
    dfA, dfT = leer("asistencias", e["archivo"]), leer("tareas", e["archivo"])
    print("Filas asistencias:", dfA.count(), "| Filas tareas:", dfT.count())
    dfA.write.mode("overwrite").saveAsTable(tabla("tb_asistencias", e["sufijo"]))
    dfT.write.mode("overwrite").saveAsTable(tabla("tb_tareas", e["sufijo"]))
    kpi_por_escala[e["filas"]] = consultas_y_kpi(e["sufijo"], f"{cap} · {e['filas']} filas · consultas")

display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

## Paso 14 · Edición, tipo de compute, runtime y versión de Spark — **CAPTURA D-09** (4.4 y 3.5.4)
Además de esta celda, capturar el selector de compute (arriba a la derecha) y, si es serverless, el panel **Environment** con su versión.

In [ ]:
marca("D-09")
def conf(clave):
    try:
        return spark.conf.get(clave)
    except Exception:
        return "no disponible en este compute"

entorno = {
    "Edición de Databricks": EDICION,
    "Versión de Spark": spark.version,
    "Runtime (DATABRICKS_RUNTIME_VERSION)": os.environ.get("DATABRICKS_RUNTIME_VERSION", "no disponible"),
    "Runtime (clusterUsageTags.sparkVersion)": conf("spark.databricks.clusterUsageTags.sparkVersion"),
    "Tipo de compute": "Serverless" if conf("spark.databricks.clusterUsageTags.clusterId") == "no disponible en este compute"
                       else conf("spark.databricks.clusterUsageTags.clusterId"),
    "Python": platform.python_version(),
    "Fecha de ejecución": ahora(),
}
for k, v in entorno.items():
    print(f"{k:42s}: {v}")
print("\nSi el tipo de compute sale 'Serverless' por descarte, confirmarlo con la captura del selector de compute.")

## Paso 15 · Medición de tiempos (4.3 y 4.6)
- **Carga** = leer el CSV del volumen + escribir la tabla Delta (`overwrite`).
- **Consulta** = el COUNT + GROUP BY del paso 12, con `collect()` para que se ejecute de verdad.
- Por cada escala y operación: **1 calentamiento (no se anota en el resumen) + 5 repeticiones**, una ejecución a la vez, mismo compute.
- Se guardan **todas** las ejecuciones (también el calentamiento, marcado) para no ocultar valores atípicos.

In [ ]:
registros = []      # una fila por ejecución
runs = []           # una fila por escala (RUN-03, RUN-04, ...)

def medir(run_id, escala, operacion, tabla_obj, consulta_txt, funcion):
    total = CALENTAMIENTO + REPETICIONES
    for i in range(total):
        es_cal = i < CALENTAMIENTO
        inicio = ahora()
        t0 = time.perf_counter()
        estado, error, filas_res = "OK", "", None
        try:
            filas_res = funcion()
        except Exception as ex:
            estado, error = "ERROR", str(ex)[:300]
        dur = time.perf_counter() - t0
        registros.append({
            "run_id": run_id, "fecha_hora": inicio, "notebook": NOTEBOOK, "escala_filas": escala,
            "operacion": operacion, "tabla": tabla_obj, "consulta": consulta_txt,
            "repeticion": 0 if es_cal else i - CALENTAMIENTO + 1, "calentamiento": es_cal,
            "duracion_s": round(dur, 4), "filas_resultado": filas_res, "estado": estado, "error": error,
        })
        print(f"{run_id} | {escala:>6} filas | {operacion:22s} | {'cal' if es_cal else 'rep ' + str(i - CALENTAMIENTO + 1)} | {dur:8.3f} s | {estado}")

for n, e in enumerate(ESCALAS):
    run_id = f"RUN-{RUN_INICIAL + n:02d}"
    inicio_run = ahora()
    for tipo in ("asistencias", "tareas"):
        t_obj = tabla(f"tb_{tipo}", e["sufijo"])
        archivo = e["archivo"]

        def carga(tipo=tipo, archivo=archivo, t_obj=t_obj):
            leer(tipo, archivo).write.mode("overwrite").saveAsTable(t_obj)
            return None
        medir(run_id, e["filas"], f"carga_{tipo}", t_obj,
              f"read csv {tipo}_{archivo}.csv + saveAsTable overwrite", carga)

        sql = CONSULTAS[tipo].format(t=t_obj)
        def consulta(sql=sql):
            return len(spark.sql(sql).collect())
        medir(run_id, e["filas"], f"consulta_{tipo}", t_obj, sql, consulta)

    errores = sum(1 for r in registros if r["run_id"] == run_id and r["estado"] != "OK")
    runs.append({"run_id": run_id, "inicio": inicio_run, "fin": ahora(), "notebook": NOTEBOOK,
                 "experimento": "EXP-02", "escala_filas": e["filas"],
                 "consultas": "carga + GROUP BY Estado_Asistencia; carga + GROUP BY Estado_Tarea",
                 "ejecuciones": CALENTAMIENTO + REPETICIONES,
                 "estado": "Completada" if errores == 0 else f"Con {errores} errores"})
print("Mediciones terminadas.")

## Paso 16 · CSV de tiempos + mediana, mínimo y máximo — **CAPTURA del CSV** (6.3, fila EXP-02)

In [ ]:
marca("Tiempos EXP-02")
df_tiempos = pd.DataFrame(registros)
df_tiempos.to_csv(f"{RUTA_RES}/tiempos_databricks.csv", index=False)

# Bytes: archivo CSV y tabla Delta
bytes_csv = {f.name: f.size for f in dbutils.fs.ls(RUTA_DATOS)}
def bytes_tabla(t):
    try:
        return spark.sql(f"DESCRIBE DETAIL {t}").select("sizeInBytes").first()[0]
    except Exception:
        return None

validas = df_tiempos[(~df_tiempos["calentamiento"]) & (df_tiempos["estado"] == "OK")]
resumen = (validas.groupby(["escala_filas", "operacion", "tabla"])["duracion_s"]
           .agg(n="count", mediana_s="median", minimo_s="min", maximo_s="max").reset_index())
resumen["filas_por_s"] = (resumen["escala_filas"] / resumen["mediana_s"]).round(0)
resumen["bytes_csv"] = resumen.apply(
    lambda r: bytes_csv.get(f"{r['operacion'].split('_')[1]}_{'base' if r['escala_filas'] == 582 else r['escala_filas']}.csv"), axis=1)
resumen["bytes_tabla"] = resumen["tabla"].map(bytes_tabla)
for c in ("mediana_s", "minimo_s", "maximo_s"):
    resumen[c] = resumen[c].round(4)
resumen.to_csv(f"{RUTA_RES}/resumen_tiempos.csv", index=False)

print("Guardado:", f"{RUTA_RES}/tiempos_databricks.csv", "y", f"{RUTA_RES}/resumen_tiempos.csv")
display(df_tiempos)
display(resumen)

## Paso 17 · Registro de ejecuciones (6.2: RUN-03, RUN-04…)

In [ ]:
marca("Registro RUN")
df_runs = pd.DataFrame(runs)
df_runs.to_csv(f"{RUTA_RES}/registro_ejecuciones.csv", index=False)
df_kpi = pd.DataFrame([{"escala_filas": k, **v} for k, v in kpi_por_escala.items()])
df_kpi.to_csv(f"{RUTA_RES}/kpi_por_escala.csv", index=False)
display(df_runs)
display(df_kpi)

## Paso 18 · Gráfico tiempo vs. cantidad de filas (3.5.5 y 6.4)

In [ ]:
import matplotlib.pyplot as plt

COLOR = {"asistencias": "#2a78d6", "tareas": "#eb6834"}
MARCADOR = {"asistencias": "o", "tareas": "s"}
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))
for eje, op, titulo in ((ejes[0], "carga", "Tiempo de carga (CSV → tabla Delta)"),
                        (ejes[1], "consulta", "Tiempo de consulta (COUNT + GROUP BY)")):
    for tipo in ("asistencias", "tareas"):
        d = resumen[resumen["operacion"] == f"{op}_{tipo}"].sort_values("escala_filas")
        if d.empty:
            continue
        err = [d["mediana_s"] - d["minimo_s"], d["maximo_s"] - d["mediana_s"]]
        eje.errorbar(d["escala_filas"], d["mediana_s"], yerr=err, color=COLOR[tipo], marker=MARCADOR[tipo],
                     markersize=8, linewidth=2, capsize=4, label=tipo.capitalize())
    eje.set_title(titulo, fontsize=11)
    eje.set_xlabel("Cantidad de filas por tabla")
    eje.set_ylabel("Tiempo (s) — mediana de 5 repeticiones")
    eje.set_xticks([e["filas"] for e in ESCALAS])
    eje.set_ylim(bottom=0)
    eje.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    for s in ("top", "right"):
        eje.spines[s].set_visible(False)
    eje.legend(frameon=False)
fig.suptitle("EXP-02 · Databricks: tiempo frente a volumen (barras = mínimo y máximo)", fontsize=12)
fig.tight_layout()
fig.savefig(f"{RUTA_RES}/grafico_tiempos.png", dpi=160)
plt.show()
print("Imagen guardada en", f"{RUTA_RES}/grafico_tiempos.png")

## Paso 19 · Descripción de lo observado (6.5)
Escribir solo lo que muestran los números del resumen. No mencionar CPU ni RAM si Databricks no las muestra (en serverless no hay Spark UI).

- ¿Cómo cambia la mediana de carga de 582 a 5 000 y a 10 000 filas? [completar con los valores medidos]
- ¿Cómo cambia la mediana de consulta? [completar]
- ¿Qué tan dispersas fueron las 5 repeticiones (mínimo–máximo)? ¿Hubo algún valor atípico? [completar]
- ¿Los KPI de la escala base coinciden con Power BI (TPA 92,10 %, ICT 56,36 %, TDT 3,44 %, IEP 75,12)? [completar]
- Límites: datos sintéticos, compute serverless sin configuración propia, posibles cachés de la plataforma; no se extrapola fuera de 582–10 000 filas.

## Paso 20 · Archivos para entregar al encargado (Anexos, E-08 y E-14)

In [ ]:
display(dbutils.fs.ls(RUTA_RES))
print("""Descargar desde Catalog → vol_rendimiento → resultados:
  tiempos_databricks.csv, resumen_tiempos.csv, registro_ejecuciones.csv, kpi_por_escala.csv, grafico_tiempos.png
Exportar el notebook: File → Export → IPython Notebook (.ipynb) y también HTML (conserva las salidas).
Adjuntar además los 6 CSV de datos y manifiesto_datos.csv / manifiesto_datos.json del generador.""")